In [1]:
import os
import json
from dotenv import load_dotenv
import anthropic
import yfinance as yf
from datetime import datetime

load_dotenv()
client = anthropic.Anthropic()
print("Setup complete")

Setup complete


In [2]:
def get_stock_price(ticker):
    """Fetch the most recent price for a stock ticker."""
    try:
        stock = yf.Ticker(ticker)
        data = stock.history(period="5d")
        if data.empty:
            return f"Error: No data found for {ticker}"
        price = data["Close"].iloc[-1]
        date = data.index[-1].date()
        return f"{ticker} most recent price: ${price:.2f} (as of {date})"
    except Exception as e:
        return f"Error fetching {ticker}: {str(e)}"


def get_historical_return(ticker, years):
    """Compute total return of a stock over N years."""
    try:
        stock = yf.Ticker(ticker)
        data = stock.history(period=f"{years + 1}y")
        if data.empty or len(data) < 2:
            return f"Error: Insufficient data for {ticker}"
        start_price = data["Close"].iloc[0]
        end_price = data["Close"].iloc[-1]
        total_return = ((end_price - start_price) / start_price) * 100
        start_date = data.index[0].date()
        end_date = data.index[-1].date()
        return f"{ticker} {years}-year return: {total_return:.2f}% (${start_price:.2f} to ${end_price:.2f}, {start_date} to {end_date})"
    except Exception as e:
        return f"Error computing return for {ticker}: {str(e)}"


def get_fundamentals(ticker):
    """Fetch key fundamental data: market cap, P/E ratio, sector, industry."""
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        
        market_cap = info.get("marketCap", None)
        market_cap_str = f"${market_cap/1e9:.1f}B" if market_cap else "N/A"
        
        pe_ratio = info.get("trailingPE", None)
        pe_str = f"{pe_ratio:.1f}x" if pe_ratio else "N/A"
        
        forward_pe = info.get("forwardPE", None)
        forward_pe_str = f"{forward_pe:.1f}x" if forward_pe else "N/A"
        
        return {
            "company_name": info.get("longName", ticker),
            "sector": info.get("sector", "N/A"),
            "industry": info.get("industry", "N/A"),
            "market_cap": market_cap_str,
            "trailing_pe": pe_str,
            "forward_pe": forward_pe_str,
            "52_week_high": f"${info.get('fiftyTwoWeekHigh', 0):.2f}",
            "52_week_low": f"${info.get('fiftyTwoWeekLow', 0):.2f}",
            "dividend_yield": f"{info.get('dividendYield', 0) * 100:.2f}%" if info.get('dividendYield') else "None"
        }
    except Exception as e:
        return f"Error fetching fundamentals for {ticker}: {str(e)}"


# Quick test
print(get_fundamentals("AVGO"))

{'company_name': 'Broadcom Inc.', 'sector': 'Technology', 'industry': 'Semiconductors', 'market_cap': '$2001.6B', 'trailing_pe': '82.6x', 'forward_pe': '23.3x', '52_week_high': '$429.31', '52_week_low': '$184.02', 'dividend_yield': '62.00%'}


In [3]:
data_tools = [
    {
        "name": "get_stock_price",
        "description": "Fetch the most recent closing price for a stock ticker.",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticker": {"type": "string", "description": "Stock ticker symbol (e.g. AAPL, NVDA)"}
            },
            "required": ["ticker"]
        }
    },
    {
        "name": "get_historical_return",
        "description": "Compute total percentage return of a stock over N years.",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticker": {"type": "string", "description": "Stock ticker symbol"},
                "years": {"type": "integer", "description": "Number of years for the return calculation"}
            },
            "required": ["ticker", "years"]
        }
    },
    {
        "name": "get_fundamentals",
        "description": "Fetch key fundamental data for a stock: market cap, P/E ratio, sector, industry, 52-week range, dividend yield.",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticker": {"type": "string", "description": "Stock ticker symbol"}
            },
            "required": ["ticker"]
        }
    }
]

print(f"Tools registered: {[t['name'] for t in data_tools]}")

Tools registered: ['get_stock_price', 'get_historical_return', 'get_fundamentals']


In [4]:
def execute_tool(name, inputs):
    """Dispatch tool calls to the right Python function."""
    if name == "get_stock_price":
        return get_stock_price(**inputs)
    elif name == "get_historical_return":
        return get_historical_return(**inputs)
    elif name == "get_fundamentals":
        return get_fundamentals(**inputs)
    else:
        return f"Error: unknown tool '{name}'"


def gather_market_data(ticker):
    """Run the agent loop to gather all market data for a ticker."""
    
    system_prompt = """You are a quantitative equity research analyst. 
    Given a stock ticker, gather comprehensive market data by calling the available tools.
    Always fetch: current price, 1-year return, 5-year return, and fundamental data.
    Be thorough and systematic."""
    
    messages = [
        {"role": "user", "content": f"Gather all available market data for {ticker}. Fetch the current price, 1-year return, 5-year return, and fundamentals."}
    ]
    
    print(f"Gathering market data for {ticker}...")
    
    while True:
        response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=2048,
            system=system_prompt,
            tools=data_tools,
            messages=messages
        )
        
        messages.append({"role": "assistant", "content": response.content})
        
        if response.stop_reason != "tool_use":
            # Extract the final summary text
            final_text = next((b.text for b in response.content if b.type == "text"), "")
            return messages, final_text
        
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"  Calling {block.name} with {block.input}")
                result = execute_tool(block.name, block.input)
                # Handle dict results from get_fundamentals
                result_str = json.dumps(result) if isinstance(result, dict) else str(result)
                print(f"  Result: {result_str[:100]}...")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result_str
                })
        
        messages.append({"role": "user", "content": tool_results})

print("Data gathering functions defined")

Data gathering functions defined


In [5]:
research_report_tool = {
    "name": "produce_research_report",
    "description": "Produce a structured equity research report based on gathered market data.",
    "input_schema": {
        "type": "object",
        "properties": {
            "ticker": {
                "type": "string",
                "description": "Stock ticker symbol"
            },
            "company_name": {
                "type": "string",
                "description": "Full company name"
            },
            "current_price": {
                "type": "number",
                "description": "Most recent stock price in USD"
            },
            "one_year_return_pct": {
                "type": "number",
                "description": "1-year total return as a percentage (e.g. 45.2 for 45.2%)"
            },
            "five_year_return_pct": {
                "type": "number",
                "description": "5-year total return as a percentage"
            },
            "market_cap": {
                "type": "string",
                "description": "Market capitalization (e.g. '$2.8T')"
            },
            "sector": {
                "type": "string",
                "description": "Company sector"
            },
            "trailing_pe": {
                "type": "string",
                "description": "Trailing P/E ratio"
            },
            "investment_thesis": {
                "type": "string",
                "description": "2-3 sentence bull case investment thesis based on the data"
            },
            "key_risks": {
                "type": "array",
                "items": {"type": "string"},
                "description": "3-4 key risks an investor should consider"
            },
            "analyst_rating": {
                "type": "string",
                "enum": ["Strong Buy", "Buy", "Hold", "Sell", "Strong Sell"],
                "description": "Analyst rating based on valuation and momentum"
            },
            "one_line_summary": {
                "type": "string",
                "description": "One punchy sentence summarizing the investment case"
            }
        },
        "required": [
            "ticker", "company_name", "current_price",
            "one_year_return_pct", "five_year_return_pct",
            "market_cap", "sector", "trailing_pe",
            "investment_thesis", "key_risks",
            "analyst_rating", "one_line_summary"
        ]
    }
}

print("Report schema defined")

Report schema defined


In [6]:
def generate_research_report(ticker, conversation_history, data_summary):
    """Use structured output to produce a formatted research report."""
    
    system_prompt = """You are a senior equity research analyst at a top investment bank.
    Based on the market data gathered, produce a structured research report.
    Be analytical, precise, and professional. Base your thesis and risks on the actual data."""
    
    # Build the prompt with all the gathered data
    prompt = f"""Based on all the market data gathered for {ticker} in this conversation, 
    produce a comprehensive structured research report. 
    Use the actual numbers from the data — do not estimate or make up figures.
    
    Data summary: {data_summary}"""
    
    # Add our prompt to the conversation history
    messages = conversation_history + [
        {"role": "user", "content": prompt}
    ]
    
    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=2048,
        system=system_prompt,
        tools=[research_report_tool],
        tool_choice={"type": "tool", "name": "produce_research_report"},
        messages=messages
    )
    
    # Extract the structured report
    tool_use = next(b for b in response.content if b.type == "tool_use")
    return tool_use.input

print("Report generator defined")

Report generator defined


In [7]:
def format_report(report):
    """Format the structured JSON report into a readable one-pager."""
    
    divider = "=" * 60
    thin_divider = "-" * 60
    
    output = f"""
{divider}
EQUITY RESEARCH REPORT
{divider}
{report['company_name']} ({report['ticker']})
Generated: {datetime.now().strftime('%B %d, %Y')}

RATING: {report['analyst_rating']}
{thin_divider}
{report['one_line_summary']}

PRICE & PERFORMANCE
{thin_divider}
Current Price:      ${report['current_price']:.2f}
1-Year Return:      {report['one_year_return_pct']:.1f}%
5-Year Return:      {report['five_year_return_pct']:.1f}%

FUNDAMENTALS
{thin_divider}
Market Cap:         {report['market_cap']}
Sector:             {report['sector']}
Trailing P/E:       {report['trailing_pe']}

INVESTMENT THESIS
{thin_divider}
{report['investment_thesis']}

KEY RISKS
{thin_divider}"""
    
    for i, risk in enumerate(report['key_risks'], 1):
        output += f"\n  {i}. {risk}"
    
    output += f"\n{divider}"
    
    return output

print("Formatter defined")

Formatter defined


In [8]:
def run_equity_research(ticker):
    """
    Full equity research pipeline:
    1. Gather market data via agent loop
    2. Generate structured report via forced tool use
    3. Format and display the one-pager
    """
    ticker = ticker.upper()
    print(f"\n{'='*60}")
    print(f"RUNNING EQUITY RESEARCH: {ticker}")
    print(f"{'='*60}\n")
    
    # Step 1: Gather data
    conversation_history, data_summary = gather_market_data(ticker)
    print(f"\nData gathering complete.")
    
    # Step 2: Generate structured report
    print(f"\nGenerating structured report...")
    report = generate_research_report(ticker, conversation_history, data_summary)
    
    # Step 3: Format and display
    formatted = format_report(report)
    print(formatted)
    
    # Also return the raw JSON for downstream use
    return report

print("Pipeline ready. Run: run_equity_research('AAPL')")

Pipeline ready. Run: run_equity_research('AAPL')


In [9]:
report = run_equity_research("AVGO")


RUNNING EQUITY RESEARCH: AVGO

Gathering market data for AVGO...
  Calling get_stock_price with {'ticker': 'AVGO'}
  Result: AVGO most recent price: $422.76 (as of 2026-04-24)...
  Calling get_historical_return with {'ticker': 'AVGO', 'years': 1}
  Result: AVGO 1-year return: 233.27% ($126.85 to $422.76, 2024-04-25 to 2026-04-24)...
  Calling get_historical_return with {'ticker': 'AVGO', 'years': 5}
  Result: AVGO 5-year return: 1703.70% ($23.44 to $422.76, 2020-04-27 to 2026-04-24)...
  Calling get_fundamentals with {'ticker': 'AVGO'}
  Result: {"company_name": "Broadcom Inc.", "sector": "Technology", "industry": "Semiconductors", "market_cap"...

Data gathering complete.

Generating structured report...

EQUITY RESEARCH REPORT
Broadcom Inc. (AVGO)
Generated: April 26, 2026

RATING: Hold
------------------------------------------------------------
A semiconductor powerhouse with exceptional momentum and AI exposure, but elevated valuation and parabolic price action warrant caution at